# Protocol resting

## Import raw data to structured JSON

Import raw resting data from `datasets/raw/resting/` to structured JSON in `datasets/resting/`.
Drop your `.txt` / `.csv` Polar export files into `cardiolab/datasets/raw/resting/` before running this cell.

In [ ]:
from cardiolab.scripts.import_rr import import_all

if __name__ == "__main__":
    import_all(protocol="resting")

## 2. Extraction des features HRV par session

Pour chaque enregistrement, les intervalles RR sont chargés et produisent **20 indicateurs** répartis en 3 domaines :

| Domaine | Indicateurs |
|---|---|
| **Temporel** | RMSSD, ln(RMSSD), SDNN, pNN50, HR moyen |
| **Fréquentiel** | VLF, LF, HF, LF/HF, HF%, LF_nu, HF_nu, HF/FC |
| **Non-linéaire** | SD1, SD2, SD1/SD2 (Poincaré) · DFA α1 (fluctuations détendues) |

In [ ]:
import glob
import json
import math

from cardiolab.protocols.resting import resting_hrv
from cardiolab.signals.rr import RRSeries

path = "cardiolab/datasets/resting/*.json"
files = sorted(glob.glob(path))

if not files:
    print("Aucun fichier JSON — déposez vos fichiers dans cardiolab/datasets/resting/")

for file in files:
    with open(file) as f:
        data = json.load(f)

    rr = RRSeries(data["rr_intervals"])
    features = resting_hrv(rr)   # retourne un HRVFeatures complet (20 champs)
    features.date = data["date"]

    print(f"\n{'═' * 54}")
    print(f"  {features.date}   ({features.duration:.0f} s enregistrés)")
    print(f"{'═' * 54}")

    print("\n  ── Domaine temporel ─────────────────────────────")
    print(f"  RMSSD      : {features.rmssd:>8.2f} ms")
    print(f"  ln(RMSSD)  : {features.ln_rmssd:>8.3f}")
    print(f"  SDNN       : {features.sdnn:>8.2f} ms")
    print(f"  pNN50      : {features.pnn50:>8.1f} %")
    print(f"  HR moyen   : {features.mean_hr:>8.1f} bpm")

    print("\n  ── Domaine fréquentiel ──────────────────────────")
    print(f"  VLF        : {features.vlf:>10.2f} ms²")
    print(f"  LF         : {features.lf:>10.2f} ms²")
    print(f"  HF         : {features.hf:>10.2f} ms²")
    print(f"  LF/HF      : {features.lf_hf:>8.3f}")
    print(f"  HF%        : {features.hf_pct:>8.3f}")
    print(f"  LF_nu      : {features.lf_nu:>8.3f}")
    print(f"  HF_nu      : {features.hf_nu:>8.3f}")
    print(f"  HF/FC      : {features.hf_hr:>8.3f}  ms²/bpm")

    print("\n  ── Non-linéaire — Poincaré + DFA ────────────────")
    print(f"  SD1        : {features.sd1:>8.2f} ms    (variabilité court terme)")
    print(f"  SD2        : {features.sd2:>8.2f} ms    (variabilité long terme)")
    print(f"  SD1/SD2    : {features.sd_ratio:>8.3f}       (normal repos : 0.25 – 0.55)")
    dfa_str = f"{features.dfa_alpha1:.3f}" if not math.isnan(features.dfa_alpha1) else "n/a (signal trop court)"
    print(f"  DFA α1     : {dfa_str:>8}       (normal repos : 0.75 – 1.25)")

In [ ]:
import glob
import json
import math

import pandas as pd
from IPython.display import display

from cardiolab.analytics.baseline import Baseline
from cardiolab.analytics.scoring import (
    readiness_score_composite,
    readiness_score_multi,
    readiness_score_nonlinear,
    readiness_score_oura,
)
from cardiolab.protocols.resting import HRVFeatures, resting_hrv
from cardiolab.signals.rr import RRSeries


# ── 1. Chargement ────────────────────────────────────────────────────

def load_features_from_json(path: str = "cardiolab/datasets/resting/*.json") -> list[HRVFeatures]:
    """Charge les fichiers JSON et retourne une liste de HRVFeatures (toutes métriques incluses)."""
    files = sorted(glob.glob(path))
    features_list = []
    for file in files:
        with open(file) as f:
            data = json.load(f)
        rr = RRSeries(data["rr_intervals"])
        features = resting_hrv(rr)   # HRVFeatures complet : pas de reconstruction manuelle
        features.date = data["date"]
        features_list.append(features)
    return features_list


# ── 2. Affichage scores ──────────────────────────────────────────────

def display_session_scores(features_list: list[HRVFeatures], baseline: Baseline) -> None:
    """Affiche les métriques clés et les 4 scores pour chaque session."""
    print("═" * 58)
    print("  RÉSULTATS PAR SESSION")
    print("═" * 58)

    for f in features_list:
        s_oura  = readiness_score_oura(f, baseline)
        s_multi = readiness_score_multi(f, baseline)
        s_nl    = readiness_score_nonlinear(f, baseline)
        s_comp  = readiness_score_composite(f, baseline)
        dfa_str = f"{f.dfa_alpha1:.3f}" if not math.isnan(f.dfa_alpha1) else "n/a"

        print(f"\n  {f.date}")
        print(f"  RMSSD  {f.rmssd:6.1f} ms  │  HR    {f.mean_hr:5.1f} bpm")
        print(f"  SD1    {f.sd1:6.2f} ms  │  SD2   {f.sd2:6.2f} ms  │  SD1/SD2 {f.sd_ratio:.3f}")
        print(f"  DFA α1  {dfa_str}")
        print(f"  {'─' * 44}")
        print(f"  Score Oura       : {s_oura:5.1f} / 100   (RMSSD 70% + HR 30%)")
        print(f"  Score Multi      : {s_multi:5.1f} / 100   (RMSSD + HR + DFA α1 + tendance)")
        print(f"  Score Non-lin.   : {s_nl:5.1f} / 100   (DFA 40% + SD1 35% + SD1/SD2 25%)")
        print(f"  Score Composite  : {s_comp:5.1f} / 100  ★ (Multi 50% + Non-lin. 50%)")

    print(f"\n{'═' * 58}")
    print("  BASELINE  (fenêtre glissante 7 sessions)")
    print(f"{'═' * 58}")
    print(f"  RMSSD médian   : {baseline.median_rmssd():.2f} ms")
    print(f"  HR moyen       : {baseline.mean_hr():.2f} bpm")
    sd1_base = baseline.median_sd1()
    dfa_base = baseline.median_dfa_alpha1()
    if sd1_base:
        print(f"  SD1 médian     : {sd1_base:.2f} ms")
    if dfa_base:
        print(f"  DFA α1 médian  : {dfa_base:.3f}")


# ── Pipeline ─────────────────────────────────────────────────────────

features_list = load_features_from_json()

if not features_list:
    print("Aucune donnée trouvée dans cardiolab/datasets/resting/")
else:
    baseline = Baseline.from_features(features_list)

    # Score composite stocké dans chaque session (utilisé par le DataFrame plus bas)
    for f in features_list:
        f.score = readiness_score_composite(f, baseline)

    display_session_scores(features_list, baseline)

### Interprétation des scores

| Score composite | Interprétation |
|---|---|
| 80 – 100 | Très bien récupéré — entraînement intensif possible |
| 60 – 80 | OK — entraînement modéré à intense |
| 40 – 60 | Fatigue modérée — privilégier la récupération |
| 20 – 40 | Fatigué — entraînement léger seulement |
| < 20 | Surcharge — repos conseillé |

### Interprétation DFA α1

| DFA α1 | Signification |
|---|---|
| 0.75 – 1.25 | Normal au repos — régulation autonome saine |
| < 0.75 | Signal de fatigue / surcompensation — vigilance |
| > 1.25 | Variabilité très structurée — rare, possible chez athlètes |

### Interprétation SD1/SD2

| SD1/SD2 | Signification |
|---|---|
| 0.25 – 0.55 | Équilibre sympatho-vagal normal au repos |
| < 0.25 | Dominance sympathique (stress, fatigue) |
| > 0.55 | Dominance parasympathique marquée |

## 4. Vue tabulaire — historique des sessions

Toutes les sessions sous forme de `DataFrame` pandas avec mise en couleur des métriques clés.

In [ ]:
# Historique complet sous forme de DataFrame pandas
# Mise en couleur : rouge = valeur basse, vert = valeur haute
if "features_list" in dir() and features_list:
    df = baseline.to_dataframe()
    cols = ["date", "rmssd", "sdnn", "mean_hr", "sd1", "sd2", "sd_ratio", "dfa_alpha1", "score"]
    display(
        df[cols].style
            .format({
                "rmssd":       "{:.1f}",
                "sdnn":        "{:.1f}",
                "mean_hr":     "{:.1f}",
                "sd1":         "{:.2f}",
                "sd2":         "{:.2f}",
                "sd_ratio":    "{:.3f}",
                "dfa_alpha1":  lambda x: f"{x:.3f}" if pd.notna(x) else "n/a",
                "score":       "{:.1f}",
            })
            .background_gradient(subset=["score"],      cmap="RdYlGn", vmin=0,    vmax=100)
            .background_gradient(subset=["rmssd"],      cmap="RdYlGn")
            .background_gradient(subset=["sd1"],        cmap="RdYlGn")
            .background_gradient(subset=["dfa_alpha1"], cmap="RdYlGn", vmin=0.5,  vmax=1.25)
            .background_gradient(subset=["mean_hr"],    cmap="RdYlGn_r")
            .set_caption("Historique des sessions — rouge = bas · vert = élevé")
    )

In [ ]:
from cardiolab.visualization.plot import plot_resting_evolution

plot_resting_evolution(path="cardiolab/datasets/resting/*.json")

# Protocol orthostatic

## Import raw data to structured JSON

Import raw orthostatic data from `datasets/raw/orthostatic/` to structured JSON in `datasets/orthostatic/`.
The recording must contain a supine phase immediately followed by a standing phase (≥ 10 min total recommended).
Drop your `.txt` / `.csv` Polar export files into `cardiolab/datasets/raw/orthostatic/` before running this cell.

In [ ]:
from cardiolab.scripts.import_rr import import_all

if __name__ == "__main__":
    import_all(protocol="orthostatic")

## 2. Affichage rapide par session

Pour chaque fichier JSON, exécute le protocole orthostatic et affiche :
- **Phase allongé** : tous les indicateurs HRV (temporel + fréquentiel + **Poincaré + DFA α1**)
- **Transition** : delta HR, HR max, durée, timestamps
- **Phase debout** : tous les indicateurs HRV (mêmes domaines)
- **Résumé** : réponse HR, variation LF/HF, variation HF%, interprétation clinique

In [ ]:
import glob
import json
import math

from cardiolab.protocols.orthostatic import orthostatic_hrv
from cardiolab.signals.rr import RRSeries

path = "cardiolab/datasets/orthostatic/*.json"
files = sorted(glob.glob(path))

for file in files:
    with open(file) as f:
        data = json.load(f)

    rr = RRSeries(data["rr_intervals"])

    try:
        result = orthostatic_hrv(rr)
    except ValueError as e:
        print(f"[SKIP] {data['date']}: {e}")
        continue

    sup = result.phases.supine.features
    std = result.phases.standing.features
    tr  = result.phases.transition

    def _fmt_dfa(val: float) -> str:
        return f"{val:.3f}" if not math.isnan(val) else "n/a"

    print(f"\n{'═' * 58}")
    print(f"  {data['date']}")
    print(f"{'═' * 58}")

    print("\n  ── PHASE ALLONGÉ ────────────────────────────────")
    print(f"  Durée    : {result.phases.supine.duration_sec:.0f} s")
    print(f"  RMSSD    : {sup.rmssd:.2f} ms  │  SDNN  : {sup.sdnn:.2f} ms  │  pNN50 : {sup.pnn50:.1f}%")
    print(f"  HR moyen : {sup.mean_hr:.1f} bpm")
    print(f"  HF       : {sup.hf:.2f} ms²   │  HF/FC : {sup.hf_hr:.3f}  │  LF/HF : {sup.lf_hf:.3f}")
    print(f"  HF_nu    : {sup.hf_nu:.3f}       │  LF_nu : {sup.lf_nu:.3f}")
    print(f"  SD1      : {sup.sd1:.2f} ms  │  SD2   : {sup.sd2:.2f} ms  │  SD1/SD2: {sup.sd_ratio:.3f}")
    print(f"  DFA α1   : {_fmt_dfa(sup.dfa_alpha1)}")

    print("\n  ── TRANSITION ───────────────────────────────────")
    print(f"  Durée    : {tr.duration_sec:.1f} s   ({tr.start_sec:.0f}s → {tr.end_sec:.0f}s)")
    print(f"  Delta HR : +{tr.delta_hr:.1f} bpm   │  HR max : {tr.peak_hr:.1f} bpm")

    print("\n  ── PHASE DEBOUT ─────────────────────────────────")
    print(f"  Durée    : {result.phases.standing.duration_sec:.0f} s")
    print(f"  RMSSD    : {std.rmssd:.2f} ms  │  SDNN  : {std.sdnn:.2f} ms  │  pNN50 : {std.pnn50:.1f}%")
    print(f"  HR moyen : {std.mean_hr:.1f} bpm")
    print(f"  HF       : {std.hf:.2f} ms²   │  HF/FC : {std.hf_hr:.3f}  │  LF/HF : {std.lf_hf:.3f}")
    print(f"  HF_nu    : {std.hf_nu:.3f}       │  LF_nu : {std.lf_nu:.3f}")
    print(f"  SD1      : {std.sd1:.2f} ms  │  SD2   : {std.sd2:.2f} ms  │  SD1/SD2: {std.sd_ratio:.3f}")
    print(f"  DFA α1   : {_fmt_dfa(std.dfa_alpha1)}")

    print("\n  ── RÉSUMÉ ORTHOSTATIQUE ─────────────────────────")
    print(f"  Réponse HR  : +{result.hr_response:.1f} bpm")
    print(f"  Δ LF/HF     :  x{result.lf_hf_ratio_change:.2f}  (> 1 = activation sympathique)")
    print(f"  Δ HF        :  {result.hf_response_pct:.1f}%   (normal : −30% à −50%)")
    print(f"  Δ HF/FC     :  {result.hf_hr_pct_change:.1f}%   (retrait vagal au lever)")
    print(f"  Résultat    :  {result.interpretation}")

## Pipeline complet (fonctions)

In [ ]:
import glob
import json
import math
from dataclasses import dataclass

import pandas as pd
from IPython.display import display

from cardiolab.protocols.orthostatic import OrthostaticResult, orthostatic_hrv
from cardiolab.signals.rr import RRSeries


@dataclass
class OrthostaticSession:
    """Conteneur pour une session orthostatique (date + résultat)."""
    date: str
    result: OrthostaticResult


def load_orthostatic_sessions(
    path: str = "cardiolab/datasets/orthostatic/*.json",
) -> list[OrthostaticSession]:
    """Charge les fichiers JSON et exécute le protocole orthostatic."""
    files = sorted(glob.glob(path))
    sessions = []
    for file in files:
        with open(file) as f:
            data = json.load(f)
        rr = RRSeries(data["rr_intervals"])
        try:
            result = orthostatic_hrv(rr)
            sessions.append(OrthostaticSession(date=data["date"], result=result))
        except ValueError as e:
            print(f"[SKIP] {data['date']}: {e}")
    return sessions


def _display_hrv_phase(label: str, features, duration_sec: float) -> None:
    """Affiche tous les indicateurs HRV d'une phase (3 domaines)."""
    dfa_str = f"{features.dfa_alpha1:.3f}" if not math.isnan(features.dfa_alpha1) else "n/a"
    print(f"\n  {label}  (durée : {duration_sec:.0f} s)")
    print(f"    ── Temporel ──────────────────────────────")
    print(f"    RMSSD    : {features.rmssd:.2f} ms")
    print(f"    ln_RMSSD : {features.ln_rmssd:.3f}")
    print(f"    SDNN     : {features.sdnn:.2f} ms")
    print(f"    pNN50    : {features.pnn50:.1f} %")
    print(f"    HR moyen : {features.mean_hr:.1f} bpm")
    print(f"    ── Fréquentiel ───────────────────────────")
    print(f"    VLF      : {features.vlf:.2f} ms²")
    print(f"    LF       : {features.lf:.2f} ms²")
    print(f"    HF       : {features.hf:.2f} ms²")
    print(f"    HF/FC    : {features.hf_hr:.3f}  ms²/bpm")
    print(f"    LF/HF    : {features.lf_hf:.3f}")
    print(f"    HF%      : {features.hf_pct:.3f}")
    print(f"    LF_nu    : {features.lf_nu:.3f}  │  HF_nu : {features.hf_nu:.3f}")
    print(f"    ── Non-linéaire ──────────────────────────")
    print(f"    SD1      : {features.sd1:.2f} ms  (court terme)")
    print(f"    SD2      : {features.sd2:.2f} ms  (long terme)")
    print(f"    SD1/SD2  : {features.sd_ratio:.3f}    (normal : 0.25 – 0.55)")
    print(f"    DFA α1   : {dfa_str}    (normal : 0.75 – 1.25)")


def display_orthostatic_results(sessions: list[OrthostaticSession]) -> None:
    """Affiche les indicateurs complets des 3 phases pour chaque session."""
    for session in sessions:
        r = session.result
        p = r.phases

        print(f"\n{'═' * 58}")
        print(f"  {session.date}")
        print(f"{'═' * 58}")

        _display_hrv_phase("ALLONGÉ",  p.supine.features,   p.supine.duration_sec)

        print(f"\n  TRANSITION  (durée : {p.transition.duration_sec:.1f} s  "
              f"│  {p.transition.start_sec:.0f}s → {p.transition.end_sec:.0f}s)")
        print(f"    Delta HR : +{p.transition.delta_hr:.1f} bpm   │  HR max : {p.transition.peak_hr:.1f} bpm")

        _display_hrv_phase("DEBOUT",  p.standing.features, p.standing.duration_sec)

        print(f"\n  {'─' * 46}")
        print(f"  RÉSUMÉ ORTHOSTATIQUE")
        print(f"  {'─' * 46}")
        print(f"  Réponse HR   : +{r.hr_response:.1f} bpm")
        print(f"  Δ LF/HF      :  x{r.lf_hf_ratio_change:.2f}  (> 1 = activation sympathique)")
        print(f"  Δ HF         :  {r.hf_response_pct:.1f}%   (normal : −30% à −50%)")
        print(f"  Δ HF/FC      :  {r.hf_hr_pct_change:.1f}%   (retrait vagal au lever)")
        print(f"  Résultat     :  {r.interpretation}")


# ── Pipeline ─────────────────────────────────────────────────────────

sessions = load_orthostatic_sessions()

if not sessions:
    print("Aucune donnée trouvée dans cardiolab/datasets/orthostatic/")
else:
    display_orthostatic_results(sessions)

    # Vue tabulaire comparative allongé vs debout
    print("\n\n" + "═" * 58)
    print("  VUE COMPARATIVE  (allongé vs debout)")
    print("═" * 58)
    rows = []
    for s in sessions:
        row = {"date": s.date}
        for col in ["rmssd", "mean_hr", "sd1", "sd2", "sd_ratio", "dfa_alpha1", "hf_nu"]:
            row[f"supine_{col}"]   = getattr(s.result.phases.supine.features, col)
            row[f"standing_{col}"] = getattr(s.result.phases.standing.features, col)
        row["hr_response"]      = s.result.hr_response
        row["hf_response_pct"]  = s.result.hf_response_pct
        row["interpretation"]   = s.result.interpretation
        rows.append(row)

    df_ortho = pd.DataFrame(rows)
    display(
        df_ortho.style
            .format({c: "{:.2f}" for c in df_ortho.select_dtypes("float").columns
                     if c not in ("supine_dfa_alpha1", "standing_dfa_alpha1")})
            .format({
                "supine_dfa_alpha1":   lambda x: f"{x:.3f}" if pd.notna(x) else "n/a",
                "standing_dfa_alpha1": lambda x: f"{x:.3f}" if pd.notna(x) else "n/a",
            })
            .background_gradient(subset=["supine_rmssd", "standing_rmssd"], cmap="RdYlGn")
            .background_gradient(subset=["hr_response"],   cmap="RdYlGn_r")
            .set_caption("Comparaison allongé / debout par session")
    )

| Résultat | Signification |
| --- | --- |
| `normal` | Réponse orthostatique normale — hausse HR 5–30 bpm |
| `elevated_response` | Réponse excessive > 30 bpm — possible POTS |
| `impaired_response` | Réponse insuffisante < 5 bpm — possible dysautonomie |
| `excessive_vagal_withdrawal` | Retrait vagal excessif — chute HF > 60 % |

# Export des données

Les fonctions du module `cardiolab.io.export` permettent d'exporter vers **CSV** et **JSON** sans dépendance à pandas.

| Format | Resting | Orthostatique |
|---|---|---|
| **CSV** | Une ligne par session, 20 colonnes | Une ligne par test, colonnes préfixées `supine_` / `transition_` / `standing_` |
| **JSON** | Tableau de sessions | Structure imbriquée complète (phases → features) |

Les fichiers sont écrits dans `cardiolab/datasets/exports/`.

In [ ]:
from pathlib import Path

from cardiolab.io.export import (
    features_to_csv,
    features_to_json,
    orthostatic_to_csv,
    orthostatic_to_json,
)

export_dir = Path("cardiolab/datasets/exports")
export_dir.mkdir(parents=True, exist_ok=True)

# ── Resting ───────────────────────────────────────────────────────────
if "features_list" in dir() and features_list:
    path_csv  = export_dir / "resting_history.csv"
    path_json = export_dir / "resting_history.json"

    features_to_csv(features_list, path_csv)
    features_to_json(features_list, path_json)

    print(f"Resting — {len(features_list)} session(s) exportée(s)")
    print(f"  CSV  → {path_csv}")
    print(f"  JSON → {path_json}")

    # Aperçu du CSV
    import pandas as pd
    from IPython.display import display
    df_check = pd.read_csv(path_csv)
    display(df_check[["date", "rmssd", "sd1", "dfa_alpha1", "score"]].head(10))
else:
    print("Aucune donnée resting en mémoire — exécutez d'abord la section Resting.")

print()

# ── Orthostatique ─────────────────────────────────────────────────────
if "sessions" in dir() and sessions:
    for session in sessions:
        date_slug = session.date.replace(":", "-").replace(" ", "_")

        path_json_o = export_dir / f"ortho_{date_slug}.json"
        path_csv_o  = export_dir / f"ortho_{date_slug}.csv"

        orthostatic_to_json(session.result, path_json_o)   # structure imbriquée complète
        orthostatic_to_csv(session.result,  path_csv_o)    # 1 ligne, colonnes préfixées

        print(f"Ortho {session.date}")
        print(f"  JSON → {path_json_o}")
        print(f"  CSV  → {path_csv_o}")
else:
    print("Aucune donnée orthostatique en mémoire — exécutez d'abord la section Orthostatic.")